# TSV counts → BigWig
Convert raw count TSV (chrom/start/end/ES_count/MS_count/LS_count/G1_count) to BigWig tracks
using the same TPM normalization as the training pipeline, so values are on the same scale
as model predictions.

In [ ]:
import numpy as np
import pandas as pd
import pyBigWig
from pathlib import Path

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
TSV_PATH   = '/Users/lzz/Downloads/rice_wt_counts_1024bp.tsv'
OUTPUT_DIR = 'output/ground_truth'

In [ ]:
df = pd.read_csv(TSV_PATH, sep='\t', dtype={'chrom': str})
if '#chrom' in df.columns:
    df = df.rename(columns={'#chrom': 'chrom'})

# The TSV from preprocess_signals.py already contains TPM-normalized values
# (columns: chrom, start, end, G1, ES, MS, LS) — no re-normalization needed.
eps = 1e-6
g1 = df['G1'].values.astype(np.float32)
es = df['ES'].values.astype(np.float32)
ms = df['MS'].values.astype(np.float32)
ls = df['LS'].values.astype(np.float32)
wrt = (0.5 * ms + ls) / (es + ms + ls + eps)

df['WRT'] = wrt

print(f'Loaded {len(df):,} bins from {TSV_PATH}')
print(df[['chrom', 'start', 'end', 'G1', 'ES', 'MS', 'LS', 'WRT']].head())

In [ ]:
chrom_sizes = (
    df.groupby('chrom')['end'].max()
    .sort_index()
    .to_dict()
)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

TRACKS = {
    'G1':  'G1',
    'ES':  'ES',
    'MS':  'MS',
    'LS':  'LS',
    'WRT': 'WRT',
}

for track, col in TRACKS.items():
    bw = pyBigWig.open(str(out_dir / f'{track}.bw'), 'w')
    bw.addHeader(list(chrom_sizes.items()))
    for chrom, grp in df.sort_values(['chrom', 'start']).groupby('chrom', sort=False):
        chroms = [chrom] * len(grp)
        starts = grp['start'].tolist()
        ends   = grp['end'].tolist()
        values = grp[col].astype(float).tolist()
        bw.addEntries(chroms, starts, ends=ends, values=values)
    bw.close()
    print(f'Written {track}.bw')

print(f'Done — {len(TRACKS)} BigWig tracks in {OUTPUT_DIR}')